# 🐾 Classificação de Cães e Gatos — Google Colab (GPU)

**Antes de executar:** Vá em `Ambiente de Execução > Alterar tipo de ambiente de execução > T4 GPU`

Melhorias em relação à versão local:
- Imagens em **224×224** (resolução nativa do VGG16)
- CNN com **BatchNormalization** e **GlobalAveragePooling**
- VGG16 com **fine-tuning em duas fases** (congela → descongela block5)
- Callbacks: EarlyStopping + ReduceLROnPlateau + ModelCheckpoint
- Resultados e modelos salvos automaticamente no **Google Drive**

## Célula 1 — Verificar GPU

In [ ]:
import tensorflow as tf

print(f'TensorFlow: {tf.__version__}')
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f'✅ GPU disponível: {gpus[0].name}')
else:
    print('⚠️  GPU NÃO detectada!')
    print('Vá em: Ambiente de Execução > Alterar tipo de ambiente de execução > T4 GPU')

## Célula 2 — Importar bibliotecas

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import shutil
import random

from sklearn.metrics import confusion_matrix, classification_report

from tensorflow.keras.preprocessing.image import ImageDataGenerator, load_img, img_to_array
from tensorflow.keras.models import Sequential, Model
from tensorflow.keras.layers import (
    Conv2D, MaxPooling2D, Flatten, Dense, Dropout,
    Input, BatchNormalization, GlobalAveragePooling2D
)
from tensorflow.keras.applications import VGG16
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau, ModelCheckpoint
from tensorflow.keras.optimizers import Adam
from google.colab import drive, files

print('✅ Bibliotecas importadas com sucesso!')

## Célula 3 — Montar Google Drive
Os modelos e resultados serão salvos automaticamente na sua pasta do Drive.

In [ ]:
drive.mount('/content/drive')

DRIVE_PATH = '/content/drive/MyDrive/classificacao_pets'
os.makedirs(f'{DRIVE_PATH}/modelos', exist_ok=True)
os.makedirs(f'{DRIVE_PATH}/saidas', exist_ok=True)

print(f'✅ Drive montado. Resultados serão salvos em: {DRIVE_PATH}')

## Célula 4 — Baixar dataset (download direto, sem Kaggle)

Dataset oficial Microsoft Cats vs Dogs (~786 MB, 25.000 imagens). Sem necessidade de conta ou token.

In [ ]:
import zipfile

URL      = 'https://download.microsoft.com/download/3/E/1/3E1C3F21-ECDB-4869-8368-6DEBA77B919F/kagglecatsanddogs_5228.zip'
ZIP_PATH = '/content/catsdogs.zip'

print('Baixando dataset (~786 MB)...')
!wget -q --show-progress "{URL}" -O "{ZIP_PATH}"

print('\nExtraindo arquivos...')
with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
    zf.extractall('/content/raw_flat')

cats_dir = '/content/raw_flat/PetImages/Cat'
dogs_dir = '/content/raw_flat/PetImages/Dog'

n_cats = len([f for f in os.listdir(cats_dir) if f.lower().endswith('.jpg')])
n_dogs = len([f for f in os.listdir(dogs_dir) if f.lower().endswith('.jpg')])

print(f'\n✅ Dataset baixado e extraído com sucesso!')
print(f'  Gatos:     {n_cats} imagens')
print(f'  Cachorros: {n_dogs} imagens')

## Célula 5 — Organizar pastas treino/validação

Separa as 25.000 imagens do dataset Microsoft (`PetImages/Cat` e `PetImages/Dog`) em `treino` (80%) e `validacao` (20%).

In [ ]:
BASE_PATH = '/content/dataset'
VAL_SPLIT = 0.2
random.seed(42)

mapa = {
    '/content/raw_flat/PetImages/Cat': 'gatos',
    '/content/raw_flat/PetImages/Dog': 'cachorros'
}

for origem, classe in mapa.items():
    os.makedirs(f'{BASE_PATH}/treino/{classe}', exist_ok=True)
    os.makedirs(f'{BASE_PATH}/validacao/{classe}', exist_ok=True)

    arquivos = [f for f in os.listdir(origem) if f.lower().endswith('.jpg')]
    random.shuffle(arquivos)
    n_val = int(len(arquivos) * VAL_SPLIT)

    for split, lista in [('validacao', arquivos[:n_val]), ('treino', arquivos[n_val:])]:
        destino = f'{BASE_PATH}/{split}/{classe}'
        if os.path.exists(destino) and len(os.listdir(destino)) > 0:
            print(f'  {split}/{classe}: já existe ({len(os.listdir(destino))} imgs), pulando.')
            continue
        for f in lista:
            try:
                shutil.copy2(os.path.join(origem, f), destino)
            except Exception:
                pass  # ignora arquivos corrompidos (poucos no dataset)
        print(f'  {split}/{classe}: {len(os.listdir(destino))} imagens copiadas')

print('\n✅ Dataset organizado!')
print(f'  Treino:    gatos={len(os.listdir(BASE_PATH+"/treino/gatos"))}, cachorros={len(os.listdir(BASE_PATH+"/treino/cachorros"))}')
print(f'  Validação: gatos={len(os.listdir(BASE_PATH+"/validacao/gatos"))}, cachorros={len(os.listdir(BASE_PATH+"/validacao/cachorros"))}')


## Célula 6 — Criar geradores de dados com data augmentation aprimorado

In [ ]:
TAMANHO_IMAGEM = (224, 224)  # Resolução nativa do VGG16
BATCH_SIZE = 32

gen_treino_aug = ImageDataGenerator(
    rescale=1./255,
    rotation_range=30,
    width_shift_range=0.2,
    height_shift_range=0.2,
    shear_range=0.2,
    zoom_range=0.25,
    horizontal_flip=True,
    brightness_range=[0.75, 1.25],
    fill_mode='nearest'
)

gen_val = ImageDataGenerator(rescale=1./255)

gerador_treino = gen_treino_aug.flow_from_directory(
    f'{BASE_PATH}/treino',
    target_size=TAMANHO_IMAGEM,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    seed=42
)

gerador_validacao = gen_val.flow_from_directory(
    f'{BASE_PATH}/validacao',
    target_size=TAMANHO_IMAGEM,
    batch_size=BATCH_SIZE,
    class_mode='binary',
    seed=42
)

print(f'Classes: {gerador_treino.class_indices}')
print(f'Treino:    {gerador_treino.samples} imagens')
print(f'Validação: {gerador_validacao.samples} imagens')

## Célula 7 — Visualizar exemplos do dataset

In [ ]:
classes = {v: k for k, v in gerador_treino.class_indices.items()}

plt.figure(figsize=(14, 7))
for batch_img, batch_label in gerador_treino:
    for j in range(min(8, BATCH_SIZE)):
        plt.subplot(2, 4, j + 1)
        plt.imshow(batch_img[j])
        classe = classes[int(round(batch_label[j]))]
        plt.title(f'{classe.capitalize()}', fontsize=12)
        plt.axis('off')
    break
plt.suptitle('Exemplos com Data Augmentation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig(f'{DRIVE_PATH}/saidas/exemplos_dataset.png', dpi=150, bbox_inches='tight')
plt.show()

## Célula 8 — Funções auxiliares de avaliação e gráficos

In [ ]:
def plotar_historico(historico, nome_modelo, caminho_saida):
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    axes[0].plot(historico.history['accuracy'],     label='Treino',    color='royalblue')
    axes[0].plot(historico.history['val_accuracy'], label='Validação', color='darkorange')
    axes[0].set_title(f'Acurácia — {nome_modelo}', fontsize=13, fontweight='bold')
    axes[0].set_xlabel('Epoch')
    axes[0].set_ylabel('Acurácia')
    axes[0].legend()
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(historico.history['loss'],     label='Treino',    color='royalblue')
    axes[1].plot(historico.history['val_loss'], label='Validação', color='darkorange')
    axes[1].set_title(f'Loss — {nome_modelo}', fontsize=13, fontweight='bold')
    axes[1].set_xlabel('Epoch')
    axes[1].set_ylabel('Loss')
    axes[1].legend()
    axes[1].grid(True, alpha=0.3)

    plt.tight_layout()
    caminho = os.path.join(caminho_saida, f'historico_{nome_modelo}.png')
    plt.savefig(caminho, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Gráfico salvo: {caminho}')


def avaliar_modelo(modelo, gerador, nome_modelo, caminho_saida):
    print(f'\nAvaliando {nome_modelo}...')
    gerador.reset()
    preds = modelo.predict(gerador, steps=gerador.samples // gerador.batch_size + 1, verbose=1)
    y_pred = (preds > 0.5).astype(int).flatten()
    y_true = gerador.classes[:len(y_pred)]
    nomes_classes = list(gerador.class_indices.keys())

    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=nomes_classes, yticklabels=nomes_classes)
    plt.title(f'Matriz de Confusão — {nome_modelo}', fontsize=13, fontweight='bold')
    plt.ylabel('Verdadeiro')
    plt.xlabel('Predito')
    caminho_cm = os.path.join(caminho_saida, f'matriz_confusao_{nome_modelo}.png')
    plt.savefig(caminho_cm, dpi=150, bbox_inches='tight')
    plt.show()

    relatorio = classification_report(y_true, y_pred, target_names=nomes_classes)
    print(f'\nRelatório de Classificação — {nome_modelo}:')
    print(relatorio)
    with open(os.path.join(caminho_saida, f'relatorio_{nome_modelo}.txt'), 'w') as f:
        f.write(relatorio)

    return (y_pred == y_true).mean()


print('✅ Funções auxiliares definidas.')

## Célula 9 — Modelo 1: CNN Personalizada com BatchNormalization

Melhorias sobre a versão local:
- **BatchNormalization** após cada convolução → treino mais estável
- **GlobalAveragePooling2D** em vez de Flatten → menos parâmetros, menos overfitting
- 4 blocos convolucionais (32 → 64 → 128 → 256 filtros)

In [ ]:
def criar_cnn_melhorada(input_shape=(224, 224, 3)):
    modelo = Sequential([
        Input(shape=input_shape),

        Conv2D(32, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2, 2),

        Conv2D(64, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2, 2),

        Conv2D(128, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2, 2),

        Conv2D(256, (3, 3), activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling2D(2, 2),

        GlobalAveragePooling2D(),
        Dense(512, activation='relu'),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ], name='cnn_personalizada')

    modelo.compile(
        optimizer=Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )
    return modelo

modelo_cnn = criar_cnn_melhorada(input_shape=(*TAMANHO_IMAGEM, 3))
modelo_cnn.summary()

## Célula 10 — Treinar CNN

In [ ]:
callbacks_cnn = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-7, verbose=1),
    ModelCheckpoint(
        filepath=f'{DRIVE_PATH}/modelos/cnn_personalizada.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print('Iniciando treinamento da CNN Personalizada...')
historico_cnn = modelo_cnn.fit(
    gerador_treino,
    steps_per_epoch=gerador_treino.samples // BATCH_SIZE,
    epochs=20,
    validation_data=gerador_validacao,
    validation_steps=gerador_validacao.samples // BATCH_SIZE,
    callbacks=callbacks_cnn
)

melhor_acc_cnn = max(historico_cnn.history['val_accuracy'])
print(f'\n✅ CNN Personalizada — Melhor val_accuracy: {melhor_acc_cnn:.4f} ({melhor_acc_cnn*100:.2f}%)')

## Célula 11 — Avaliar CNN

In [ ]:
plotar_historico(historico_cnn, 'CNN_Personalizada', f'{DRIVE_PATH}/saidas')
acc_cnn = avaliar_modelo(modelo_cnn, gerador_validacao, 'CNN_Personalizada', f'{DRIVE_PATH}/saidas')
print(f'\nAcurácia final CNN: {acc_cnn*100:.2f}%')

## Célula 12 — Modelo 2: VGG16 — Fase 1 (base congelada)

Fase 1: todas as camadas VGG16 congeladas, treina só as camadas densas do topo.

In [ ]:
base_vgg16 = VGG16(weights='imagenet', include_top=False, input_shape=(*TAMANHO_IMAGEM, 3))

for layer in base_vgg16.layers:
    layer.trainable = False

x = base_vgg16.output
x = GlobalAveragePooling2D()(x)
x = Dense(256, activation='relu')(x)
x = Dropout(0.5)(x)
saida = Dense(1, activation='sigmoid')(x)

modelo_vgg16 = Model(inputs=base_vgg16.input, outputs=saida, name='vgg16_transfer')
modelo_vgg16.compile(
    optimizer=Adam(learning_rate=1e-3),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print(f'Camadas treináveis (Fase 1): {sum(1 for l in modelo_vgg16.layers if l.trainable)}')

callbacks_fase1 = [
    EarlyStopping(monitor='val_loss', patience=4, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=2, min_lr=1e-7, verbose=1),
]

print('\nFase 1: Treinando camadas densas (base congelada)...')
historico_fase1 = modelo_vgg16.fit(
    gerador_treino,
    steps_per_epoch=gerador_treino.samples // BATCH_SIZE,
    epochs=10,
    validation_data=gerador_validacao,
    validation_steps=gerador_validacao.samples // BATCH_SIZE,
    callbacks=callbacks_fase1
)

print(f'\nFase 1 concluída — val_accuracy: {max(historico_fase1.history["val_accuracy"]):.4f}')

## Célula 13 — VGG16 Fase 2: Fine-tuning do block5

Descongela as últimas camadas convolucionais e retreina com learning rate muito baixo → **+3–5% de acurácia**.

In [ ]:
for layer in base_vgg16.layers:
    layer.trainable = layer.name.startswith('block5')

modelo_vgg16.compile(
    optimizer=Adam(learning_rate=1e-5),
    loss='binary_crossentropy',
    metrics=['accuracy']
)

print(f'Camadas treináveis (Fase 2): {sum(1 for l in modelo_vgg16.layers if l.trainable)}')

callbacks_fase2 = [
    EarlyStopping(monitor='val_loss', patience=5, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.5, patience=3, min_lr=1e-8, verbose=1),
    ModelCheckpoint(
        filepath=f'{DRIVE_PATH}/modelos/vgg16_finetuned.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

print('\nFase 2: Fine-tuning block5 (LR=1e-5)...')
historico_fase2 = modelo_vgg16.fit(
    gerador_treino,
    steps_per_epoch=gerador_treino.samples // BATCH_SIZE,
    epochs=10,
    validation_data=gerador_validacao,
    validation_steps=gerador_validacao.samples // BATCH_SIZE,
    callbacks=callbacks_fase2
)

melhor_acc_vgg = max(historico_fase2.history['val_accuracy'])
print(f'\n✅ VGG16 Fine-tuned — Melhor val_accuracy: {melhor_acc_vgg:.4f} ({melhor_acc_vgg*100:.2f}%)')

## Célula 14 — Avaliar VGG16

In [ ]:
historico_vgg_completo = type('H', (), {
    'history': {
        'accuracy':     historico_fase1.history['accuracy']     + historico_fase2.history['accuracy'],
        'val_accuracy': historico_fase1.history['val_accuracy'] + historico_fase2.history['val_accuracy'],
        'loss':         historico_fase1.history['loss']         + historico_fase2.history['loss'],
        'val_loss':     historico_fase1.history['val_loss']     + historico_fase2.history['val_loss'],
    }
})()

plotar_historico(historico_vgg_completo, 'VGG16_FineTuned', f'{DRIVE_PATH}/saidas')
acc_vgg = avaliar_modelo(modelo_vgg16, gerador_validacao, 'VGG16_FineTuned', f'{DRIVE_PATH}/saidas')
print(f'\nAcurácia final VGG16: {acc_vgg*100:.2f}%')

## Célula 15 — Comparação final dos modelos

In [ ]:
print('=' * 50)
print('      COMPARAÇÃO FINAL DOS MODELOS')
print('=' * 50)
print(f'  CNN Personalizada:  {acc_cnn*100:.2f}%')
print(f'  VGG16 Fine-tuned:   {acc_vgg*100:.2f}%')
print('=' * 50)
melhor = 'VGG16 Fine-tuned' if acc_vgg > acc_cnn else 'CNN Personalizada'
print(f'  Melhor modelo: {melhor}')

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(['CNN Personalizada', 'VGG16 Fine-tuned'],
              [acc_cnn * 100, acc_vgg * 100],
              color=['royalblue', 'darkorange'], width=0.5, edgecolor='white')
for bar, acc in zip(bars, [acc_cnn * 100, acc_vgg * 100]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
            f'{acc:.2f}%', ha='center', va='bottom', fontsize=13, fontweight='bold')
ax.set_ylim(0, 110)
ax.set_ylabel('Acurácia (%)', fontsize=12)
ax.set_title('Comparação de Acurácia — Validação', fontsize=14, fontweight='bold')
ax.axhline(y=90, color='green', linestyle='--', alpha=0.5, label='meta 90%')
ax.legend()
ax.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.savefig(f'{DRIVE_PATH}/saidas/comparacao_modelos.png', dpi=150, bbox_inches='tight')
plt.show()

## Célula 16 — Testar com uma imagem própria

In [ ]:
from PIL import Image
import io

print('Faça upload de uma foto de cachorro ou gato:')
uploaded = files.upload()

for nome_arquivo, conteudo in uploaded.items():
    img = Image.open(io.BytesIO(conteudo)).convert('RGB')
    arr = np.array(img.resize(TAMANHO_IMAGEM)) / 255.0
    arr = np.expand_dims(arr, axis=0)

    pred_cnn = modelo_cnn.predict(arr, verbose=0)[0][0]
    pred_vgg = modelo_vgg16.predict(arr, verbose=0)[0][0]

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))
    for ax, pred, nome in zip(axes, [pred_cnn, pred_vgg], ['CNN Personalizada', 'VGG16 Fine-tuned']):
        ax.imshow(img)
        ax.axis('off')
        if pred > 0.5:
            resultado, cor = f'🐕 Cachorro ({pred*100:.1f}%)', 'deepskyblue'
        else:
            resultado, cor = f'🐈 Gato ({(1-pred)*100:.1f}%)', 'tomato'
        ax.set_title(f'{nome}\n{resultado}', fontsize=13, fontweight='bold', color=cor)

    plt.suptitle(f'Arquivo: {nome_arquivo}', fontsize=11)
    plt.tight_layout()
    plt.savefig(f'{DRIVE_PATH}/saidas/predicao_{nome_arquivo}.png', dpi=150, bbox_inches='tight')
    plt.show()

## Célula 17 — Listar arquivos salvos no Drive

In [ ]:
print(f'Arquivos salvos em {DRIVE_PATH}:')
for raiz, dirs, arquivos in os.walk(DRIVE_PATH):
    nivel = raiz.replace(DRIVE_PATH, '').count(os.sep)
    indent = '  ' * nivel
    print(f'{indent}{os.path.basename(raiz)}/')
    for arquivo in arquivos:
        tamanho = os.path.getsize(os.path.join(raiz, arquivo)) / 1024
        print(f'{indent}  {arquivo}  ({tamanho:.0f} KB)')